# GDELT Mentions — Preliminary Validation and EDA

## 0. Setup

In [ ]:
!pip -q install duckdb pyarrow

from pathlib import Path
import glob
import os
import shutil

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from IPython.display import display
from google.colab import drive

SEED = 42
MEMORY_LIMIT = '3GB'
THREADS = 2

plt.rcParams.update({
    'figure.dpi': 115,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'axes.axisbelow': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
})

def human_number(x, pos=None):
    if abs(x) >= 1_000_000:
        return f'{x / 1_000_000:.1f}M'
    if abs(x) >= 1_000:
        return f'{x / 1_000:.0f}K'
    return f'{x:.0f}'

HUMAN = mtick.FuncFormatter(human_number)

drive.mount('/content/drive')

ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Data-GDELT'),
    Path('/content/drive/MyDrive/Data - GDELT'),
]
DATA_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find Data-GDELT in MyDrive. Update ROOT_CANDIDATES.')

TMP_DIR = Path('/content/duckdb_tmp')
TMP_DIR.mkdir(exist_ok=True)

con = duckdb.connect()
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{TMP_DIR}'")
con.execute("SET preserve_insertion_order=false")

print('Drive root:', DATA_ROOT)
print('DuckDB memory limit:', MEMORY_LIMIT)
print('Local free disk:', f"{shutil.disk_usage('/content').free / 1e9:.1f} GB")

Mounted at /content/drive


FileNotFoundError: Could not find Data-GDELT in MyDrive. Update ROOT_CANDIDATES.

In [ ]:
def q(name):
    return '"' + name.replace('"', '""') + '"'


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def raw_profile(columns):
    expressions = []
    for c in columns:
        col = q(c)
        expressions.extend([
            f"SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {q(c + '__null')}",
            f"SUM(CASE WHEN {col} IS NOT NULL AND TRIM({col}) = '' THEN 1 ELSE 0 END) AS {q(c + '__empty')}",
            f"APPROX_COUNT_DISTINCT(NULLIF(TRIM({col}), '')) AS {q(c + '__unique')}",
        ])

    row = con.execute(
        'SELECT COUNT(*) AS rows, ' + ', '.join(expressions) + ' FROM raw'
    ).fetchdf().iloc[0]

    total = int(row['rows'])
    records = []
    for c in columns:
        nulls = int(row[f'{c}__null'])
        empty = int(row[f'{c}__empty'])
        records.append({
            'column': c,
            'null_count': nulls,
            'empty_count': empty,
            'missing_or_empty_pct': 100 * (nulls + empty) / total if total else 0,
            'approx_unique': int(row[f'{c}__unique']),
        })

    return total, pd.DataFrame(records)


def plot_missing_profile(profile, title):
    p = profile.sort_values('missing_or_empty_pct')
    fig, ax = plt.subplots(figsize=(9, max(3.6, 0.34 * len(p))))
    ax.barh(p['column'], p['missing_or_empty_pct'])
    ax.set_xlabel('Null or empty fields (%)')
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()


def numeric_validation(specs):
    expressions = []
    for c, spec in specs.items():
        dtype = spec['dtype']
        low = spec.get('low')
        high = spec.get('high')
        col = q(c)
        parsed = f"TRY_CAST(NULLIF(TRIM({col}), '') AS {dtype})"
        nonempty = f"{col} IS NOT NULL AND TRIM({col}) <> ''"

        expressions.extend([
            f"SUM(CASE WHEN {nonempty} AND {parsed} IS NULL THEN 1 ELSE 0 END) AS {q(c + '__parse_fail')}",
            f"SUM(CASE WHEN {parsed} = 0 THEN 1 ELSE 0 END) AS {q(c + '__zero')}",
            f"MIN({parsed}) AS {q(c + '__min')}",
            f"MAX({parsed}) AS {q(c + '__max')}",
        ])

        if low is not None and high is not None:
            expressions.append(
                f"SUM(CASE WHEN {parsed} IS NOT NULL AND ({parsed} < {low} OR {parsed} > {high}) THEN 1 ELSE 0 END) "
                f"AS {q(c + '__out_of_range')}"
            )
        else:
            expressions.append(f"0 AS {q(c + '__out_of_range')}")

    row = con.execute('SELECT ' + ', '.join(expressions) + ' FROM raw').fetchdf().iloc[0]
    records = []
    for c in specs:
        records.append({
            'column': c,
            'parse_failures': int(row[f'{c}__parse_fail']),
            'zeros': int(row[f'{c}__zero']),
            'min': row[f'{c}__min'],
            'max': row[f'{c}__max'],
            'out_of_range': int(row[f'{c}__out_of_range']),
        })
    return pd.DataFrame(records)


def timestamp_validation(column, fmt):
    col = q(column)
    parsed = f"TRY_STRPTIME(NULLIF(TRIM({col}), ''), '{fmt}')"
    nonempty = f"{col} IS NOT NULL AND TRIM({col}) <> ''"
    return con.execute(f'''
        SELECT
            SUM(CASE WHEN {nonempty} AND {parsed} IS NULL THEN 1 ELSE 0 END) AS parse_failures,
            MIN({parsed}) AS min_time,
            MAX({parsed}) AS max_time
        FROM raw
    ''').fetchdf()


def time_coverage(column, fmt, title):
    col = q(column)
    parsed = f"TRY_STRPTIME(NULLIF(TRIM({col}), ''), '{fmt}')"

    daily = con.execute(f'''
        WITH valid AS (
            SELECT {parsed} AS ts
            FROM raw
            WHERE {parsed} IS NOT NULL
        )
        SELECT
            CAST(ts AS DATE) AS date,
            COUNT(*) AS records,
            COUNT(DISTINCT ts) AS slots_seen
        FROM valid
        GROUP BY 1
        ORDER BY 1
    ''').fetchdf()

    daily['date'] = pd.to_datetime(daily['date'])
    expected_dates = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
    missing_dates = expected_dates.difference(pd.DatetimeIndex(daily['date']))
    partial = daily[daily['slots_seen'] < 96].copy()

    full_days = daily[daily['slots_seen'] == 96]
    baseline = full_days['records'].median() if len(full_days) else daily['records'].median()
    peak = daily.loc[daily['records'].idxmax()]

    print('Missing calendar dates:', len(missing_dates))
    if len(missing_dates):
        display(pd.DataFrame({'missing_date': missing_dates}))
    print('Dates with fewer than 96 observed 15-minute timestamps:', len(partial))
    if len(partial):
        display(partial)
    print(
        'Peak day:', peak['date'].date(),
        f"({int(peak['records']):,} records; {peak['records'] / baseline:.2f}x median full-day volume)"
    )

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(daily['date'], daily['records'], marker='o', linewidth=1.5)
    if len(daily) >= 7:
        ax.plot(daily['date'], daily['records'].rolling(7, min_periods=3).mean(), linewidth=2, label='7-day average')
        ax.legend(frameon=False)
    ax.set_ylabel('Records')
    ax.set_title(title)
    ax.yaxis.set_major_formatter(HUMAN)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    files = con.execute(f'''
        WITH valid AS (
            SELECT source_file, {parsed} AS ts
            FROM raw
            WHERE {parsed} IS NOT NULL
        )
        SELECT
            source_file,
            MIN(ts) AS start_time,
            MAX(ts) AS end_time,
            COUNT(*) AS records,
            COUNT(DISTINCT ts) AS slots_seen
        FROM valid
        GROUP BY 1
        ORDER BY start_time
    ''').fetchdf()

    files['file'] = files['source_file'].map(lambda x: '/'.join(Path(x).parts[-2:]))
    display(files[['file', 'start_time', 'end_time', 'records', 'slots_seen']])

    overlaps = []
    for i in range(len(files)):
        for j in range(i + 1, len(files)):
            start = max(files.loc[i, 'start_time'], files.loc[j, 'start_time'])
            end = min(files.loc[i, 'end_time'], files.loc[j, 'end_time'])
            if pd.notna(start) and pd.notna(end) and start <= end:
                overlaps.append({
                    'file_1': files.loc[i, 'file'],
                    'file_2': files.loc[j, 'file'],
                    'overlap_start': start,
                    'overlap_end': end,
                })

    overlap_df = pd.DataFrame(overlaps)
    print('File pairs with overlapping time ranges:', len(overlap_df))
    if len(overlap_df):
        display(overlap_df)

    return daily, files, overlap_df

## 1. Build the local slim Parquet

The selected Mentions fields are copied to local Parquet without parsing or row removal. `source_file` is retained so repeated records and partition overlap can be checked later.

In [ ]:
DATASET_FOLDER = 'Mentions'
LOCAL_PARQUET = Path('/content/gdelt_mentions_slim.parquet')
FORCE_REBUILD = False

KEEP_COLUMNS = ['GLOBALEVENTID', 'EventTimeDate', 'MentionTimeDate', 'MentionType', 'MentionSourceName', 'MentionIdentifier', 'Confidence', 'InRawText', 'MentionDocLen', 'MentionDocTone', 'MentionDocTranslationInfo']
CSV_FILES = sorted(glob.glob(str(DATA_ROOT / DATASET_FOLDER / '**' / '*.csv'), recursive=True))
if not CSV_FILES:
    raise FileNotFoundError(f'No CSV files found under {DATA_ROOT / DATASET_FOLDER}')

header = pd.read_csv(CSV_FILES[0], nrows=0).columns.tolist()
missing_columns = [c for c in KEEP_COLUMNS if c not in header]
if missing_columns:
    raise ValueError(f'Requested columns not found in first CSV: {missing_columns}')

raw_gb = sum(Path(f).stat().st_size for f in CSV_FILES) / 1e9
print(f'CSV files: {len(CSV_FILES)} | raw size: {raw_gb:.2f} GB')

if FORCE_REBUILD or not LOCAL_PARQUET.exists():
    files_sql = '[' + ', '.join(sql_string(f) for f in CSV_FILES) + ']'
    selected = ',\n        '.join(f'CAST({q(c)} AS VARCHAR) AS {q(c)}' for c in KEEP_COLUMNS)
    con.execute(f'''
        COPY (
            SELECT
                {selected},
                filename AS source_file
            FROM read_csv(
                {files_sql},
                header=true,
                all_varchar=true,
                union_by_name=true,
                filename=true
            )
        )
        TO '{LOCAL_PARQUET}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
    ''')

print('Local Parquet:', LOCAL_PARQUET)
print('Parquet size:', f"{LOCAL_PARQUET.stat().st_size / 1e9:.2f} GB")

con.execute(f"CREATE OR REPLACE VIEW raw AS SELECT * FROM read_parquet('{LOCAL_PARQUET}')")
print('Rows in Parquet:', f"{con.execute('SELECT COUNT(*) FROM raw').fetchone()[0]:,}")

## 2. Schema, cardinality, missing fields, and empty fields

In [ ]:
row_count, mentions_profile = raw_profile(KEEP_COLUMNS)
print('Rows:', f'{row_count:,}')
display(mentions_profile)
plot_missing_profile(mentions_profile, 'Mentions: missing or empty fields')

`MentionDocTranslationInfo` is provenance metadata. A blank value does not by itself indicate a defective record, so it is reported separately rather than imputed.

## 3. Numeric and timestamp parsing validation

In [ ]:
MENTION_NUMERIC = {
    'MentionType': {'dtype': 'BIGINT', 'low': 1, 'high': 6},
    'Confidence': {'dtype': 'BIGINT', 'low': 10, 'high': 100},
    'InRawText': {'dtype': 'BIGINT', 'low': 0, 'high': 1},
    'MentionDocTone': {'dtype': 'DOUBLE', 'low': -100, 'high': 100},
}
display(numeric_validation(MENTION_NUMERIC))

length_check = con.execute('''
    SELECT
        SUM(CASE WHEN MentionDocLen IS NOT NULL AND TRIM(MentionDocLen) <> ''
                  AND TRY_CAST(NULLIF(TRIM(MentionDocLen), '') AS BIGINT) IS NULL THEN 1 ELSE 0 END) AS parse_failures,
        MIN(TRY_CAST(NULLIF(TRIM(MentionDocLen), '') AS BIGINT)) AS min,
        MAX(TRY_CAST(NULLIF(TRIM(MentionDocLen), '') AS BIGINT)) AS max,
        SUM(CASE WHEN TRY_CAST(NULLIF(TRIM(MentionDocLen), '') AS BIGINT) < 0 THEN 1 ELSE 0 END) AS negative_values
    FROM raw
''').fetchdf()
display(length_check)

print('EventTimeDate')
display(timestamp_validation('EventTimeDate', '%Y%m%d%H%M%S'))
print('MentionTimeDate')
display(timestamp_validation('MentionTimeDate', '%Y%m%d%H%M%S'))

confidence_values = con.execute('''
    SELECT TRY_CAST(NULLIF(TRIM(Confidence), '') AS BIGINT) AS confidence, COUNT(*) AS rows
    FROM raw
    WHERE TRY_CAST(NULLIF(TRIM(Confidence), '') AS BIGINT) IS NOT NULL
    GROUP BY 1
    ORDER BY 1
''').fetchdf()
display(confidence_values)

## 4. Unit of analysis and repeated keys

In [ ]:
overview = con.execute('''
    SELECT
        COUNT(*) AS rows,
        APPROX_COUNT_DISTINCT(NULLIF(TRIM(GLOBALEVENTID), '')) AS approx_events,
        APPROX_COUNT_DISTINCT(NULLIF(TRIM(MentionIdentifier), '')) AS approx_documents,
        APPROX_COUNT_DISTINCT(NULLIF(TRIM(MentionSourceName), '')) AS approx_sources
    FROM raw
''').fetchdf()
display(overview)

event_repeat = con.execute('''
    WITH g AS (
        SELECT
            NULLIF(TRIM(GLOBALEVENTID), '') AS event_id,
            COUNT(*) AS rows,
            COUNT(DISTINCT source_file) AS files
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
        GROUP BY 1
    )
    SELECT
        COUNT(*) AS distinct_event_ids,
        SUM(CASE WHEN rows > 1 THEN 1 ELSE 0 END) AS repeated_event_ids,
        SUM(CASE WHEN files > 1 THEN 1 ELSE 0 END) AS event_ids_across_files
    FROM g
''').fetchdf()
display(event_repeat)

document_repeat = con.execute('''
    WITH g AS (
        SELECT
            NULLIF(TRIM(MentionIdentifier), '') AS document_id,
            COUNT(*) AS rows,
            COUNT(DISTINCT NULLIF(TRIM(GLOBALEVENTID), '')) AS events,
            COUNT(DISTINCT source_file) AS files
        FROM raw
        WHERE NULLIF(TRIM(MentionIdentifier), '') IS NOT NULL
        GROUP BY 1
    )
    SELECT
        COUNT(*) AS distinct_documents,
        SUM(CASE WHEN rows > 1 THEN 1 ELSE 0 END) AS repeated_documents,
        SUM(CASE WHEN events > 1 THEN 1 ELSE 0 END) AS documents_with_multiple_events,
        SUM(CASE WHEN files > 1 THEN 1 ELSE 0 END) AS documents_across_files,
        MAX(rows) AS max_rows_for_one_document
    FROM g
''').fetchdf()
display(document_repeat)

In [ ]:
# Apparent exact repeats under the selected export fields.
content_signature = "MD5(CONCAT_WS('|', " + ', '.join(
    f"COALESCE({q(c)}, '<NULL>')" for c in KEEP_COLUMNS
) + "))"

apparent_duplicates = con.execute(f'''
    WITH g AS (
        SELECT
            {content_signature} AS row_signature,
            COUNT(*) AS rows,
            COUNT(DISTINCT source_file) AS files
        FROM raw
        GROUP BY 1
    )
    SELECT
        SUM(CASE WHEN rows > 1 THEN 1 ELSE 0 END) AS repeated_signature_groups,
        SUM(CASE WHEN rows > 1 THEN rows - 1 ELSE 0 END) AS surplus_rows,
        SUM(CASE WHEN rows > 1 AND files > 1 THEN 1 ELSE 0 END) AS repeated_groups_across_files
    FROM g
''').fetchdf()
display(apparent_duplicates)

event_document_pairs = con.execute('''
    WITH g AS (
        SELECT
            NULLIF(TRIM(GLOBALEVENTID), '') AS event_id,
            NULLIF(TRIM(MentionIdentifier), '') AS document_id,
            COUNT(*) AS rows,
            COUNT(DISTINCT source_file) AS files
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
          AND NULLIF(TRIM(MentionIdentifier), '') IS NOT NULL
        GROUP BY 1, 2
    )
    SELECT
        COUNT(*) AS event_document_pairs,
        SUM(CASE WHEN rows > 1 THEN 1 ELSE 0 END) AS repeated_pairs,
        SUM(CASE WHEN rows > 1 AND files > 1 THEN 1 ELSE 0 END) AS repeated_pairs_across_files
    FROM g
''').fetchdf()
display(event_document_pairs)

Mentions does not have a single row identifier in this reduced export. Repeated event IDs and repeated document URLs are expected. Apparent identical rows are **not automatically deleted** because positional fields such as sentence and character offsets are not present here and can distinguish mentions in the full schema.

## 5. Time coverage, daily spikes, partial dates, and file overlap

In [ ]:
mention_daily, mention_file_ranges, mention_overlaps = time_coverage(
    'MentionTimeDate', '%Y%m%d%H%M%S', 'Mentions: daily record volume'
)

## 6. Source and document structure

In [ ]:
mention_types = con.execute('''
    SELECT TRY_CAST(NULLIF(TRIM(MentionType), '') AS BIGINT) AS mention_type, COUNT(*) AS rows
    FROM raw
    GROUP BY 1
    ORDER BY 1
''').fetchdf()
display(mention_types)

source_counts = con.execute('''
    SELECT MentionSourceName AS source, COUNT(*) AS mentions
    FROM raw
    WHERE NULLIF(TRIM(MentionSourceName), '') IS NOT NULL
    GROUP BY 1
    ORDER BY mentions DESC
    LIMIT 20
''').fetchdf()
display(source_counts)

fig, ax = plt.subplots(figsize=(9, 6))
plot_df = source_counts.sort_values('mentions')
ax.barh(plot_df['source'], plot_df['mentions'])
ax.set_xlabel('Mention records')
ax.set_title('Top 20 sources by mention volume')
ax.xaxis.set_major_formatter(HUMAN)
plt.tight_layout()
plt.show()

In [ ]:
events_per_article = con.execute('''
    WITH article AS (
        SELECT
            MentionIdentifier,
            COUNT(DISTINCT NULLIF(TRIM(GLOBALEVENTID), '')) AS events
        FROM raw
        WHERE NULLIF(TRIM(MentionIdentifier), '') IS NOT NULL
        GROUP BY 1
    )
    SELECT
        CASE
            WHEN events = 1 THEN '1 event'
            WHEN events BETWEEN 2 AND 5 THEN '2-5 events'
            WHEN events BETWEEN 6 AND 10 THEN '6-10 events'
            ELSE '>10 events'
        END AS group_name,
        COUNT(*) AS documents
    FROM article
    GROUP BY 1
''').fetchdf()

order = ['1 event', '2-5 events', '6-10 events', '>10 events']
events_per_article['group_name'] = pd.Categorical(events_per_article['group_name'], order, ordered=True)
events_per_article = events_per_article.sort_values('group_name')
events_per_article['percent'] = 100 * events_per_article['documents'] / events_per_article['documents'].sum()
display(events_per_article)

## 7. Confidence and extraction quality

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(confidence_values['confidence'].astype(str), confidence_values['rows'])
ax.set_xlabel('Confidence score')
ax.set_ylabel('Mention records')
ax.set_title('GDELT extraction confidence')
ax.yaxis.set_major_formatter(HUMAN)
plt.tight_layout()
plt.show()

confidence_by_raw = con.execute('''
    SELECT
        TRY_CAST(NULLIF(TRIM(InRawText), '') AS BIGINT) AS in_raw_text,
        COUNT(*) AS rows,
        AVG(TRY_CAST(NULLIF(TRIM(Confidence), '') AS DOUBLE)) AS mean_confidence,
        MEDIAN(TRY_CAST(NULLIF(TRIM(Confidence), '') AS DOUBLE)) AS median_confidence
    FROM raw
    GROUP BY 1
    ORDER BY 1
''').fetchdf()
display(confidence_by_raw)

## 8. Document length and article tone

In [ ]:
doc_length = con.execute('''
    SELECT
        QUANTILE_CONT(x, 0.25) AS q1,
        MEDIAN(x) AS median,
        QUANTILE_CONT(x, 0.75) AS q3,
        QUANTILE_CONT(x, 0.95) AS p95,
        MAX(x) AS max
    FROM (
        SELECT TRY_CAST(NULLIF(TRIM(MentionDocLen), '') AS DOUBLE) AS x
        FROM raw
    )
    WHERE x IS NOT NULL
''').fetchdf()

display(doc_length)
q1 = float(doc_length.loc[0, 'q1'])
q3 = float(doc_length.loc[0, 'q3'])
iqr = q3 - q1
upper = q3 + 1.5 * iqr

long_docs = con.execute(f'''
    SELECT COUNT(*) AS rows_above_iqr_upper
    FROM raw
    WHERE TRY_CAST(NULLIF(TRIM(MentionDocLen), '') AS DOUBLE) > {upper}
''').fetchdf()
print(f'IQR: {iqr:,.1f}')
print(f'Upper IQR fence: {upper:,.1f}')
display(long_docs)

The IQR rule is used only to describe the long upper tail. Long documents, transcripts, and roundups can be legitimate records, so they are retained.

In [ ]:
tone_sample = con.execute('''
    SELECT TRY_CAST(NULLIF(TRIM(MentionDocTone), '') AS DOUBLE) AS tone
    FROM raw
    WHERE TRY_CAST(NULLIF(TRIM(MentionDocTone), '') AS DOUBLE) IS NOT NULL
      AND MOD(HASH(COALESCE(GLOBALEVENTID, '') || '|' || COALESCE(MentionIdentifier, '')), 1000) < 100
    LIMIT 200000
''').fetchdf()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(tone_sample['tone'], bins=70)
ax.set_xlabel('Mention document tone')
ax.set_ylabel('Sampled mention records')
ax.set_title('MentionDocTone distribution')
plt.tight_layout()
plt.show()

In [ ]:
translation = con.execute('''
    SELECT
        SUM(CASE WHEN NULLIF(TRIM(MentionDocTranslationInfo), '') IS NULL THEN 1 ELSE 0 END) AS no_translation_metadata,
        SUM(CASE WHEN NULLIF(TRIM(MentionDocTranslationInfo), '') IS NOT NULL THEN 1 ELSE 0 END) AS has_translation_metadata,
        COUNT(*) AS rows
    FROM raw
''').fetchdf()
translation['translation_metadata_pct'] = 100 * translation['has_translation_metadata'] / translation['rows']
display(translation)

## 9. Media-attention distribution

In [ ]:
attention_groups = con.execute('''
    WITH event_counts AS (
        SELECT GLOBALEVENTID, COUNT(*) AS mentions
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
        GROUP BY 1
    )
    SELECT
        CASE
            WHEN mentions = 1 THEN '1 mention'
            WHEN mentions BETWEEN 2 AND 5 THEN '2-5 mentions'
            WHEN mentions BETWEEN 6 AND 10 THEN '6-10 mentions'
            WHEN mentions BETWEEN 11 AND 50 THEN '11-50 mentions'
            WHEN mentions BETWEEN 51 AND 100 THEN '51-100 mentions'
            ELSE '>100 mentions'
        END AS group_name,
        COUNT(*) AS events
    FROM event_counts
    GROUP BY 1
''').fetchdf()

attention_order = ['1 mention', '2-5 mentions', '6-10 mentions', '11-50 mentions', '51-100 mentions', '>100 mentions']
attention_groups['group_name'] = pd.Categorical(attention_groups['group_name'], attention_order, ordered=True)
attention_groups = attention_groups.sort_values('group_name')
attention_groups['percent_of_events'] = 100 * attention_groups['events'] / attention_groups['events'].sum()
display(attention_groups)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(attention_groups['group_name'].astype(str), attention_groups['percent_of_events'])
ax.set_ylabel('Events (%)')
ax.set_title('Distribution of mention volume across events')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
concentration = con.execute('''
    WITH event_counts AS (
        SELECT GLOBALEVENTID, COUNT(*) AS mentions
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
        GROUP BY 1
    ),
    ranked AS (
        SELECT
            mentions,
            ROW_NUMBER() OVER (ORDER BY mentions DESC) AS rn,
            COUNT(*) OVER () AS n_events,
            SUM(mentions) OVER () AS total_mentions
        FROM event_counts
    )
    SELECT
        SUM(CASE WHEN rn <= CEIL(n_events * 0.01) THEN mentions ELSE 0 END) * 100.0 / MAX(total_mentions) AS top_1_pct,
        SUM(CASE WHEN rn <= CEIL(n_events * 0.05) THEN mentions ELSE 0 END) * 100.0 / MAX(total_mentions) AS top_5_pct,
        SUM(CASE WHEN rn <= CEIL(n_events * 0.10) THEN mentions ELSE 0 END) * 100.0 / MAX(total_mentions) AS top_10_pct
    FROM ranked
''').fetchdf()
display(concentration)

In [ ]:
log_attention = con.execute('''
    WITH event_counts AS (
        SELECT GLOBALEVENTID, COUNT(*) AS mentions
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
        GROUP BY 1
    )
    SELECT ROUND(LN(1 + mentions), 1) AS log_mentions, COUNT(*) AS events
    FROM event_counts
    GROUP BY 1
    ORDER BY 1
''').fetchdf()

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(log_attention['log_mentions'], log_attention['events'], width=0.09)
ax.set_xlabel('log(1 + mentions per event)')
ax.set_ylabel('Events')
ax.set_title('Log-transformed event-attention distribution')
ax.yaxis.set_major_formatter(HUMAN)
plt.tight_layout()
plt.show()

## 10. Story age and observed event duration

In [ ]:
story_age = con.execute('''
    WITH x AS (
        SELECT
            DATE_DIFF(
                'minute',
                TRY_STRPTIME(NULLIF(TRIM(EventTimeDate), ''), '%Y%m%d%H%M%S'),
                TRY_STRPTIME(NULLIF(TRIM(MentionTimeDate), ''), '%Y%m%d%H%M%S')
            ) / 60.0 AS age_hours
        FROM raw
    )
    SELECT
        SUM(CASE WHEN age_hours < 0 THEN 1 ELSE 0 END) AS negative_age_rows,
        MEDIAN(age_hours) AS median_hours,
        QUANTILE_CONT(age_hours, 0.75) AS q3_hours,
        QUANTILE_CONT(age_hours, 0.95) AS p95_hours,
        QUANTILE_CONT(age_hours, 0.99) AS p99_hours
    FROM x
    WHERE age_hours IS NOT NULL
''').fetchdf()
display(story_age)

age_groups = con.execute('''
    WITH x AS (
        SELECT
            DATE_DIFF(
                'minute',
                TRY_STRPTIME(NULLIF(TRIM(EventTimeDate), ''), '%Y%m%d%H%M%S'),
                TRY_STRPTIME(NULLIF(TRIM(MentionTimeDate), ''), '%Y%m%d%H%M%S')
            ) / 60.0 AS age_hours
        FROM raw
    )
    SELECT
        CASE
            WHEN age_hours < 1 THEN '<1h'
            WHEN age_hours < 6 THEN '1-6h'
            WHEN age_hours < 12 THEN '6-12h'
            WHEN age_hours < 24 THEN '12-24h'
            WHEN age_hours < 48 THEN '1-2d'
            WHEN age_hours < 72 THEN '2-3d'
            WHEN age_hours < 168 THEN '3-7d'
            ELSE '>7d'
        END AS age_group,
        COUNT(*) AS mentions
    FROM x
    WHERE age_hours >= 0
    GROUP BY 1
''').fetchdf()

age_order = ['<1h', '1-6h', '6-12h', '12-24h', '1-2d', '2-3d', '3-7d', '>7d']
age_groups['age_group'] = pd.Categorical(age_groups['age_group'], age_order, ordered=True)
age_groups = age_groups.sort_values('age_group')
age_groups['percent'] = 100 * age_groups['mentions'] / age_groups['mentions'].sum()
display(age_groups)

In [ ]:
duration_summary = con.execute('''
    WITH event_life AS (
        SELECT
            GLOBALEVENTID,
            MIN(TRY_STRPTIME(NULLIF(TRIM(MentionTimeDate), ''), '%Y%m%d%H%M%S')) AS first_mention,
            MAX(TRY_STRPTIME(NULLIF(TRIM(MentionTimeDate), ''), '%Y%m%d%H%M%S')) AS last_mention,
            COUNT(*) AS mentions
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
        GROUP BY 1
    ),
    x AS (
        SELECT DATE_DIFF('minute', first_mention, last_mention) / 60.0 AS duration_hours, mentions
        FROM event_life
    )
    SELECT
        COUNT(*) AS events,
        MEDIAN(duration_hours) AS median_duration_hours,
        QUANTILE_CONT(duration_hours, 0.75) AS q3_duration_hours,
        QUANTILE_CONT(duration_hours, 0.95) AS p95_duration_hours,
        QUANTILE_CONT(duration_hours, 0.99) AS p99_duration_hours
    FROM x
''').fetchdf()
display(duration_summary)

Duration here is **observed duration inside the extracted window**. Events near the beginning or end of the window may be censored, so this is not yet treated as a final attention lifespan measure.

## 11. Source breadth and multivariate relationships

In [ ]:
event_sample = con.execute('''
    SELECT *
    FROM (
        SELECT
            GLOBALEVENTID,
            COUNT(*) AS mention_records,
            COUNT(DISTINCT NULLIF(TRIM(MentionSourceName), '')) AS unique_sources,
            COUNT(DISTINCT NULLIF(TRIM(MentionIdentifier), '')) AS unique_documents
        FROM raw
        WHERE NULLIF(TRIM(GLOBALEVENTID), '') IS NOT NULL
        GROUP BY 1
    )
    WHERE MOD(HASH(COALESCE(GLOBALEVENTID, '')), 1000) < 20
    LIMIT 50000
''').fetchdf()

display(event_sample[['mention_records', 'unique_sources', 'unique_documents']].describe(percentiles=[.5, .75, .9, .95, .99]))

fig, ax = plt.subplots(figsize=(7.5, 5.5))
h = ax.hexbin(
    np.log1p(event_sample['mention_records']),
    np.log1p(event_sample['unique_sources']),
    gridsize=45,
    bins='log',
    mincnt=1,
)
ax.set_xlabel('log(1 + mention records)')
ax.set_ylabel('log(1 + unique sources)')
ax.set_title('Mention volume vs source breadth')
fig.colorbar(h, ax=ax, label='log10(events per hexagon)')
plt.tight_layout()
plt.show()

In [ ]:
tone_by_age = con.execute('''
    WITH x AS (
        SELECT
            DATE_DIFF(
                'minute',
                TRY_STRPTIME(NULLIF(TRIM(EventTimeDate), ''), '%Y%m%d%H%M%S'),
                TRY_STRPTIME(NULLIF(TRIM(MentionTimeDate), ''), '%Y%m%d%H%M%S')
            ) / 60.0 AS age_hours,
            TRY_CAST(NULLIF(TRIM(MentionDocTone), '') AS DOUBLE) AS tone
        FROM raw
    )
    SELECT
        CASE
            WHEN age_hours < 1 THEN '<1h'
            WHEN age_hours < 6 THEN '1-6h'
            WHEN age_hours < 12 THEN '6-12h'
            WHEN age_hours < 24 THEN '12-24h'
            WHEN age_hours < 48 THEN '1-2d'
            WHEN age_hours < 72 THEN '2-3d'
            WHEN age_hours < 168 THEN '3-7d'
            ELSE '>7d'
        END AS age_group,
        COUNT(*) AS mentions,
        AVG(tone) AS mean_tone,
        MEDIAN(tone) AS median_tone
    FROM x
    WHERE age_hours >= 0 AND tone IS NOT NULL
    GROUP BY 1
''').fetchdf()

tone_by_age['age_group'] = pd.Categorical(tone_by_age['age_group'], age_order, ordered=True)
tone_by_age = tone_by_age.sort_values('age_group')
display(tone_by_age)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(tone_by_age['age_group'].astype(str), tone_by_age['median_tone'], marker='o')
ax.axhline(0, linewidth=1)
ax.set_xlabel('Story age')
ax.set_ylabel('Median article tone')
ax.set_title('Article tone across story age')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 12. Preliminary analysis decisions

- The Drive CSVs are read once to create `/content/gdelt_mentions_slim.parquet`; all checks after that scan local Parquet.
- DuckDB is memory-limited and may spill large group-bys to `/content/duckdb_tmp` instead of exhausting RAM.
- Raw fields are preserved as text; parse failures are measured explicitly.
- Apparent duplicate rows are reported but not automatically removed because this reduced export does not contain positional mention fields.
- Long documents and extreme attention values are retained during preliminary analysis.
- Event duration is labeled observed duration because the extraction window can censor trajectories.